# Silver - Construccion de tabla enriquecida
Este notebook crea los objetos de la capa Silver para analitica y consumo posterior en Gold.

Parametro esperado en Databricks SQL: catalogo.

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "silver")
dbutils.widgets.text("storageName", "adlsproyectoje")

## Tabla Silver: POKEMON_ENRIQUECIDO
**Objetivo:** consolidar datos de Pokemon con atributos de negocio (zona, tipos, clasificaciones) para analisis en Power BI.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.${esquema}.POKEMON_ENRIQUECIDO
AS
WITH base AS (
  SELECT
    p.pokemon_id,
    p.numero_pokedex,
    p.nombre AS pokemon_nombre,
    p.generacion_id,
    g.numero_generacion,
    g.descripcion AS generacion_descripcion,
    '01' AS zona,
    p.tipo1_id,
    t1.nombre_tipo AS tipo_primario,
    p.tipo2_id,
    t2.nombre_tipo AS tipo_secundario,
    p.tipo_experiencia_id,
    et.nombre_tipo_experiencia,
    p.experiencia_nivel_100,
    p.evolucion_final,
    p.tasa_captura,
    p.legendario,
    p.mega_evolucion,
    p.forma_alola,
    p.forma_galar,
    p.ps,
    p.ataque,
    p.defensa,
    p.ataque_especial,
    p.defensa_especial,
    p.velocidad,
    p.total_base,
    p.media,
    p.desviacion_estandar,
    p.altura_m,
    p.peso_kg,
    p.imc
  FROM ${catalogo}.bronze.POKEMON p
  LEFT JOIN ${catalogo}.bronze.GENERATIONS g
    ON p.generacion_id = g.generacion_id
  LEFT JOIN ${catalogo}.bronze.TYPES t1
    ON p.tipo1_id = t1.tipo_id
  LEFT JOIN ${catalogo}.bronze.TYPES t2
    ON p.tipo2_id = t2.tipo_id
  LEFT JOIN ${catalogo}.bronze.EXPERIENCE_TYPES et
    ON p.tipo_experiencia_id = et.tipo_experiencia_id
)
SELECT
  *,
  CASE WHEN legendario = 1 THEN 'Legendario' ELSE 'No legendario' END AS categoria_legendario,
  CASE WHEN mega_evolucion = 1 THEN 'Mega' ELSE 'Base' END AS categoria_mega
FROM base;

## Diccionario de datos Silver
**Tabla:** silver.POKEMON_ENRIQUECIDO
- **Grano:** 1 fila por pokemon_id.
- **Claves:** pokemon_id, numero_pokedex.
- **Dimensiones:** zona, numero_generacion, tipo_primario, tipo_secundario, nombre_tipo_experiencia, categoria_legendario, categoria_mega.
- **Metricas:** total_base, ataque, defensa, ataque_especial, defensa_especial, velocidad, tasa_captura.
- **Uso en BI:** base de detalle para segmentaciones y como fuente de agregaciones Gold.

In [0]:
%sql
SELECT COUNT(*) AS total_filas FROM ${catalogo}.${esquema}.POKEMON_ENRIQUECIDO;